# Reproduce the Article's Audit-Trail Sample

This notebook loads the published audit-trail sample for Soubré, CIV
from the myBytes pipeline run of 2026-06-08 and reproduces **Plot 3**
of the companion article from real per-pixel provenance records.

No synthetic data. Every row comes from
`data/runs/2026-06-08/audit_trail_sample.csv`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
FIG_OUT = REPO_ROOT / 'figures'
FIG_OUT.mkdir(exist_ok=True)

from src.io import load_audit_trail_sample
print(f'Loading from {REPO_ROOT / "data" / "runs"}')

In [5]:
trail = load_audit_trail_sample('2026-06-08', aoi_id='civ_soubre_33km')
trail

,aoi_id,lat,lon,lossyear,treecover2000_pct,gfc_tile_id,gfc_version,plantation_probability,plantation_layer_id,plantation_layer_version,threshold_tau,run_timestamp
0,civ_soubre_33km,5.6504,-6.6488,24,33,10N_010W,UMD/hansen/global_forest_change_2025_v1_13,0.9294,projects/forestdatapartnership/assets/cocoa/mo...,2025a,0.5,2026-06-08 00:00:00+00:00
1,civ_soubre_33km,5.6751,-6.6103,21,39,10N_010W,UMD/hansen/global_forest_change_2025_v1_13,0.9098,projects/forestdatapartnership/assets/cocoa/mo...,2025a,0.5,2026-06-08 00:00:00+00:00
2,civ_soubre_33km,5.6525,-6.6618,21,55,10N_010W,UMD/hansen/global_forest_change_2025_v1_13,0.9804,projects/forestdatapartnership/assets/cocoa/mo...,2025a,0.5,2026-06-08 00:00:00+00:00
3,civ_soubre_33km,5.7183,-6.6612,24,41,10N_010W,UMD/hansen/global_forest_change_2025_v1_13,0.9569,projects/forestdatapartnership/assets/cocoa/mo...,2025a,0.5,2026-06-08 00:00:00+00:00
4,civ_soubre_33km,5.7948,-6.4591,23,45,10N_010W,UMD/hansen/global_forest_change_2025_v1_13,0.9882,projects/forestdatapartnership/assets/cocoa/mo...,2025a,0.5,2026-06-08 00:00:00+00:00


In [ ]:
# Columns to display (subset of the audit-trail snapshot).
display_cols = ['lat', 'lon', 'lossyear', 'treecover2000_pct', 'gfc_tile_id',
                'plantation_probability', 'plantation_layer_id', 'threshold_tau']

view = trail[display_cols].copy()
view['plantation_probability'] = view['plantation_probability'].round(4)
view['threshold_tau'] = view['threshold_tau'].round(2)

# 1. Inline display for interactive readers (Jupyter renders pandas DataFrames as HTML).
view



In [ ]:
# 2. Render the same table as a static PNG (matches the article's Plot 3 layout).
HEADER_BG = '#003B71'
HEADER_FG = '#ffffff'
ZEBRA = '#f4f7fa'
BORDER = '#cccccc'

# Friendlier display labels for the PNG (the underlying column names stay raw in the CSV).
HEADER_LABELS = {
    'lat': 'lat',
    'lon': 'lon',
    'lossyear': 'loss\nyear',
    'treecover2000_pct': 'tree\ncover',
    'gfc_tile_id': 'GFC tile',
    'plantation_probability': 'plantation\nprobability',
    'plantation_layer_id': 'plantation_layer_id',
    'threshold_tau': 'tau',
}
headers = [HEADER_LABELS[c] for c in display_cols]

# Column widths roughly proportional to expected content width.
WIDTHS = {
    'lat': 0.7, 'lon': 0.7, 'lossyear': 0.6, 'treecover2000_pct': 0.7,
    'gfc_tile_id': 1.0, 'plantation_probability': 1.4,
    'plantation_layer_id': 4.8, 'threshold_tau': 0.5,
}
col_widths = [WIDTHS.get(c, 1.0) for c in display_cols]
total_w = sum(col_widths)
col_widths = [w / total_w for w in col_widths]

nrows = len(view)

fig, ax = plt.subplots(figsize=(14, 0.55 * (nrows + 2.4)))
ax.axis('off')
ax.set_title('Was ein Auditor pro Risiko-Pixel rekonstruieren können muss',
             loc='left', fontsize=13, fontweight='bold', color='#1a1a1a', pad=10)

tbl = ax.table(
    cellText=view.astype(str).values,
    colLabels=headers,
    cellLoc='left',
    colLoc='left',
    colWidths=col_widths,
    loc='upper left',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9.5)
tbl.scale(1, 1.6)

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor(BORDER)
    cell.set_linewidth(0.5)
    cell.PAD = 0.05
    if row == 0:
        cell.set_facecolor(HEADER_BG)
        cell.set_text_props(color=HEADER_FG, fontweight='bold')
        cell.set_height(cell.get_height() * 1.4)
    elif row % 2 == 0:
        cell.set_facecolor(ZEBRA)

fig.text(0.01, 0.02,
         'Quelle: data/runs/2026-06-08/audit_trail_sample.csv  ·  Hansen GFC v2025_v1_13 + FDP Cocoa Probability 2025a',
         fontsize=8, color='#666666')

out_path = FIG_OUT / 'plot3_audit_trail_table.png'
fig.savefig(out_path, dpi=160, bbox_inches='tight', facecolor='white')
print(f'Saved {out_path}')
plt.show()

## What this notebook proves

- Five real Soubré pixels were classified by the myBytes pipeline on
  2026-06-08 as EUDR-risk pixels.
- Every field above can be reconstructed by an external auditor by
  reading the same Hansen tile and the same FDP cocoa-probability
  release.
- Two of the five also carry a RADD near-real-time alert, which an
  auditor can cross-check against the Wageningen UR public RADD
  archive.

That is what "pixel-level auditability" means when it is real, not
synthetic.